In [ ]:
#!/usr/bin/env python3
"""
══════════════════════════════════════════════════════════════════════
🌻 KI-Labor: Deutsche Word-Embeddings aus ConceptNet Numberbatch
══════════════════════════════════════════════════════════════════════

Dieses Skript extrahiert deutsche Wort-Vektoren aus ConceptNet
Numberbatch, reduziert die Dimensionen per PCA und exportiert
eine kompakte JSON-Datei für den Browser-Einsatz.

VORAUSSETZUNGEN:
    pip install numpy scikit-learn

AUSFÜHRUNG:
    python extract_german_embeddings.py

    Das Skript lädt automatisch die Numberbatch-Daten herunter
    (~180 MB komprimiert), filtert deutsche Einträge, reduziert
    die Dimensionen und speichert das Ergebnis als JSON.

AUSGABE:
    german_embeddings.json     - Kompakte Embeddings (~2-4 MB)
    german_embeddings_meta.json - Metadaten & Statistiken

AUTOR: KI-Lehrerfortbildung Stufe 2
LIZENZ: CC-BY-SA 4.0 (Numberbatch), MIT (dieses Skript)
══════════════════════════════════════════════════════════════════════
"""

import json
import gzip
import os
import sys
import re
import urllib.request
from pathlib import Path

import numpy as np
from sklearn.decomposition import PCA

# ══════════════════════════════════════════════════════════════════
# KONFIGURATION
# ══════════════════════════════════════════════════════════════════

CONFIG = {
    # Numberbatch Download-URL (multilingual, ~180 MB gzip)
    "url": "https://conceptnet.s3.amazonaws.com/downloads/2019/numberbatch/numberbatch-19.08.txt.gz",
    
    # Alternative: Ältere Version (kleiner, ~130 MB)
    # "url": "https://conceptnet.s3.amazonaws.com/downloads/2017/numberbatch/numberbatch-17.06.txt.gz",
    
    # Lokaler Cache-Pfad
    "cache_file": "numberbatch-19.08.txt.gz",
    
    # Sprache filtern
    "language": "de",
    
    # Maximale Anzahl Wörter (nach Häufigkeits-Ranking)
    "max_words": 5000,
    
    # Ziel-Dimensionen nach PCA (Original: 300)
    "target_dims": 100,
    
    # Ausgabe-Dateien
    "output_json": "german_embeddings.json",
    "output_meta": "german_embeddings_meta.json",
    
    # Nur "saubere" Wörter: keine Sonderzeichen, Zahlen, etc.
    "clean_words_only": True,
    
    # Mindestlänge der Wörter
    "min_word_length": 2,
    
    # Dezimalstellen für JSON-Export (weniger = kleiner)
    "decimal_places": 4,
}

# ══════════════════════════════════════════════════════════════════
# ZUSÄTZLICHE PFLICHTWÖRTER
# ══════════════════════════════════════════════════════════════════
# Diese Wörter werden auf jeden Fall eingeschlossen (falls im
# Modell vorhanden), auch wenn sie nicht im Top-N sind.
# Nützlich für didaktische Szenarien und Analogie-Aufgaben.

MUST_INCLUDE = {
    # Tiere
    "hund", "katze", "maus", "pferd", "vogel", "fisch", "bär",
    "löwe", "tiger", "elefant", "affe", "schlange", "adler",
    
    # Familie & Personen
    "mann", "frau", "kind", "junge", "mädchen", "mutter", "vater",
    "bruder", "schwester", "onkel", "tante", "oma", "opa",
    "könig", "königin", "prinz", "prinzessin",
    
    # Essen & Trinken
    "brot", "wasser", "milch", "apfel", "kuchen", "eis", "käse",
    "fleisch", "gemüse", "obst", "kartoffel", "tomate", "zucker",
    
    # Natur & Wetter
    "sonne", "mond", "stern", "regen", "schnee", "wind", "wolke",
    "baum", "blume", "wald", "berg", "meer", "fluss", "see",
    
    # Gebäude & Orte
    "haus", "schule", "kirche", "stadt", "dorf", "straße",
    "krankenhaus", "bibliothek", "kino", "rathaus",
    
    # Länder & Städte
    "deutschland", "frankreich", "italien", "spanien", "england",
    "berlin", "paris", "london", "rom", "wien", "münchen",
    
    # Berufe
    "arzt", "lehrer", "polizist", "bauer", "koch", "pilot",
    
    # Farben
    "rot", "blau", "grün", "gelb", "schwarz", "weiß",
    
    # Gefühle
    "glück", "trauer", "angst", "liebe", "freude", "wut",
    
    # Adjektive
    "groß", "klein", "schnell", "langsam", "warm", "kalt",
    "gut", "schlecht", "alt", "neu", "jung", "schön",
    
    # Verben (Grundformen)
    "laufen", "essen", "trinken", "schlafen", "spielen",
    "lernen", "lesen", "schreiben", "denken", "sprechen",
    
    # Transport
    "auto", "fahrrad", "zug", "flugzeug", "schiff", "bus",
    
    # Musik
    "musik", "gitarre", "klavier", "lied", "tanz",
    
    # Sport
    "fußball", "schwimmen", "tennis",
    
    # Technik & Wissenschaft
    "computer", "telefon", "internet", "roboter",
    "mathematik", "physik", "chemie", "biologie",
    
    # Eisverkäufer-Szenario
    "temperatur", "verkauf", "sommer", "winter",
    
    # Sonnenblumen-Szenario
    "sonnenblume", "wachstum", "pflanze", "garten",
}


# ══════════════════════════════════════════════════════════════════
# HILFSFUNKTIONEN
# ══════════════════════════════════════════════════════════════════

def download_with_progress(url, filepath):
    """Datei herunterladen mit Fortschrittsanzeige."""
    print(f"\n📥 Lade herunter: {url}")
    print(f"   Ziel: {filepath}")
    
    req = urllib.request.Request(url)
    req.add_header("User-Agent", "KI-Labor-Embedding-Extractor/1.0")
    
    response = urllib.request.urlopen(req)
    total = int(response.headers.get("Content-Length", 0))
    
    downloaded = 0
    chunk_size = 1024 * 1024  # 1 MB
    
    with open(filepath, "wb") as f:
        while True:
            chunk = response.read(chunk_size)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                pct = downloaded / total * 100
                mb = downloaded / (1024 * 1024)
                total_mb = total / (1024 * 1024)
                print(f"\r   {mb:.1f} / {total_mb:.1f} MB ({pct:.1f}%)", end="", flush=True)
            else:
                mb = downloaded / (1024 * 1024)
                print(f"\r   {mb:.1f} MB heruntergeladen...", end="", flush=True)
    
    print(f"\n   ✅ Download abgeschlossen ({downloaded / (1024*1024):.1f} MB)")


def is_clean_word(word):
    """Prüft, ob ein Wort 'sauber' ist (keine Sonderzeichen etc.)."""
    if len(word) < CONFIG["min_word_length"]:
        return False
    # Nur Buchstaben (inkl. deutsche Umlaute und ß)
    return bool(re.match(r'^[a-zäöüß]+$', word))


def load_german_vectors(filepath):
    """Deutsche Vektoren aus Numberbatch-Datei laden."""
    print(f"\n📖 Lese Vektoren aus {filepath}...")
    
    prefix = f"/c/{CONFIG['language']}/"
    vectors = {}
    total_lines = 0
    
    opener = gzip.open if filepath.endswith(".gz") else open
    
    with opener(filepath, "rt", encoding="utf-8") as f:
        # Erste Zeile: Dimensionen
        header = f.readline().strip().split()
        if len(header) == 2:
            num_words, dims = int(header[0]), int(header[1])
            print(f"   Modell: {num_words:,} Wörter, {dims} Dimensionen")
        
        for line in f:
            total_lines += 1
            if total_lines % 200000 == 0:
                print(f"\r   {total_lines:,} Zeilen gelesen, {len(vectors):,} deutsche Wörter...", end="", flush=True)
            
            # Nur deutsche Einträge
            if not line.startswith(prefix):
                continue
            
            parts = line.strip().split(" ")
            if len(parts) < 10:  # Mindestens einige Dimensionen
                continue
            
            # Wort extrahieren: /c/de/wort -> wort
            word = parts[0][len(prefix):]
            
            # Unterstriche durch Leerzeichen (Mehrwort-Ausdrücke)
            word = word.replace("_", " ")
            
            # Nur Einzelwörter (keine Mehrwort-Ausdrücke) für Spiel
            if " " in word:
                continue
            
            # Sauberkeits-Check
            if CONFIG["clean_words_only"] and not is_clean_word(word):
                continue
            
            # Vektor parsen
            try:
                vec = np.array([float(x) for x in parts[1:]], dtype=np.float32)
                vectors[word] = vec
            except ValueError:
                continue
    
    print(f"\n   ✅ {len(vectors):,} deutsche Wörter geladen")
    return vectors


def select_top_words(vectors, max_words):
    """Die wichtigsten Wörter auswählen."""
    print(f"\n🔍 Wähle die {max_words} wichtigsten Wörter aus...")
    
    # Numberbatch ist nach Häufigkeit sortiert, also nehmen wir
    # einfach die ersten max_words Einträge.
    # Aber: MUST_INCLUDE-Wörter werden vorrangig eingeschlossen.
    
    must_have = {}
    regular = {}
    
    for word, vec in vectors.items():
        if word in MUST_INCLUDE:
            must_have[word] = vec
        else:
            regular[word] = vec
    
    # Auffüllen bis max_words
    remaining_slots = max_words - len(must_have)
    
    selected = dict(must_have)
    for word, vec in regular.items():
        if remaining_slots <= 0:
            break
        selected[word] = vec
        remaining_slots -= 1
    
    found_must = len(must_have)
    missing_must = MUST_INCLUDE - set(must_have.keys())
    
    print(f"   Pflicht-Wörter gefunden: {found_must}/{len(MUST_INCLUDE)}")
    if missing_must:
        print(f"   ⚠️  Nicht im Modell: {', '.join(sorted(missing_must)[:20])}")
        if len(missing_must) > 20:
            print(f"       ... und {len(missing_must) - 20} weitere")
    print(f"   ✅ {len(selected):,} Wörter ausgewählt")
    
    return selected


def reduce_dimensions(vectors, target_dims):
    """PCA-Dimensionsreduktion."""
    print(f"\n📐 PCA: {len(list(vectors.values())[0])} → {target_dims} Dimensionen...")
    
    words = list(vectors.keys())
    matrix = np.array([vectors[w] for w in words])
    
    # Normalisieren (Zero-Mean)
    mean = matrix.mean(axis=0)
    matrix_centered = matrix - mean
    
    # PCA
    pca = PCA(n_components=target_dims)
    matrix_reduced = pca.fit_transform(matrix_centered)
    
    # Varianz-Erklärung
    explained = sum(pca.explained_variance_ratio_) * 100
    print(f"   Erklärte Varianz: {explained:.1f}%")
    
    # L2-Normalisierung (für Cosinus-Ähnlichkeit)
    norms = np.linalg.norm(matrix_reduced, axis=1, keepdims=True)
    norms[norms == 0] = 1  # Division durch Null vermeiden
    matrix_normalized = matrix_reduced / norms
    
    result = {w: matrix_normalized[i] for i, w in enumerate(words)}
    
    print(f"   ✅ Reduktion abgeschlossen")
    return result, explained, pca


def export_json(vectors, explained_variance, output_path, meta_path):
    """Als JSON exportieren."""
    print(f"\n💾 Exportiere nach {output_path}...")
    
    dp = CONFIG["decimal_places"]
    
    # Hauptdatei: {wort: [v1, v2, ...], ...}
    export = {}
    for word, vec in sorted(vectors.items()):
        export[word] = [round(float(v), dp) for v in vec]
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(export, f, ensure_ascii=False, separators=(",", ":"))
    
    file_size = os.path.getsize(output_path)
    print(f"   Dateigröße: {file_size / (1024*1024):.2f} MB")
    print(f"   Wörter: {len(export):,}")
    print(f"   Dimensionen: {len(list(export.values())[0])}")
    
    # Metadaten
    meta = {
        "source": "ConceptNet Numberbatch 19.08",
        "license": "CC-BY-SA 4.0",
        "language": CONFIG["language"],
        "num_words": len(export),
        "dimensions": CONFIG["target_dims"],
        "original_dimensions": 300,
        "pca_explained_variance_pct": round(float(explained_variance), 2),
        "decimal_places": dp,
        "file_size_bytes": file_size,
        "description": "Deutsche Word-Embeddings für das KI-Labor (Semantris-Spiel)",
    }
    
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    
    print(f"   ✅ Export abgeschlossen")
    return meta


def test_embeddings(vectors):
    """Embedding-Qualität mit Beispiel-Abfragen testen."""
    print("\n🧪 Qualitätstest: Ähnlichste Wörter\n")
    
    words = list(vectors.keys())
    matrix = np.array([vectors[w] for w in words])
    
    def most_similar(word, n=8):
        if word not in vectors:
            return f"   '{word}' nicht im Vokabular"
        
        query = vectors[word]
        # Cosinus-Ähnlichkeit (Vektoren sind bereits normalisiert)
        sims = matrix @ query
        
        # Top-N (ohne das Wort selbst)
        indices = np.argsort(sims)[::-1]
        results = []
        for idx in indices:
            w = words[idx]
            if w != word:
                results.append((w, float(sims[idx])))
            if len(results) >= n:
                break
        return results
    
    def analogy(a, b, c):
        """a ist zu b wie c ist zu ???"""
        if a not in vectors or b not in vectors or c not in vectors:
            missing = [w for w in [a, b, c] if w not in vectors]
            return f"   Nicht gefunden: {', '.join(missing)}"
        
        # vec(b) - vec(a) + vec(c) ≈ vec(?)
        result_vec = vectors[b] - vectors[a] + vectors[c]
        result_vec = result_vec / np.linalg.norm(result_vec)
        
        sims = matrix @ result_vec
        indices = np.argsort(sims)[::-1]
        
        exclude = {a, b, c}
        results = []
        for idx in indices:
            w = words[idx]
            if w not in exclude:
                results.append((w, float(sims[idx])))
            if len(results) >= 5:
                break
        return results
    
    # Ähnlichkeits-Tests
    test_words = ["hund", "sonne", "berlin", "arzt", "musik", "auto"]
    for tw in test_words:
        result = most_similar(tw)
        if isinstance(result, str):
            print(result)
        else:
            similar_str = ", ".join(f"{w}({s:.3f})" for w, s in result[:5])
            print(f"   {tw:15s} → {similar_str}")
    
    # Analogie-Tests
    print("\n🧪 Analogie-Test: A ist zu B wie C ist zu ???\n")
    analogies = [
        ("mann", "könig", "frau", "→ königin?"),
        ("deutschland", "berlin", "frankreich", "→ paris?"),
        ("hund", "katze", "groß", "→ klein?"),
    ]
    for a, b, c, label in analogies:
        result = analogy(a, b, c)
        if isinstance(result, str):
            print(f"   {a} → {b}, {c} → ? {result}")
        else:
            top = ", ".join(f"{w}({s:.3f})" for w, s in result[:3])
            print(f"   {a} → {b}, {c} → ? {label}")
            print(f"   {'':15s}   Ergebnis: {top}")
    
    print()


# ══════════════════════════════════════════════════════════════════
# HAUPTPROGRAMM
# ══════════════════════════════════════════════════════════════════

def main():
    print("═" * 65)
    print("🌻 KI-Labor: Deutsche Word-Embeddings Extraktor")
    print("   ConceptNet Numberbatch → Kompaktes JSON für den Browser")
    print("═" * 65)
    
    cache = Path(CONFIG["cache_file"])
    
    # 1. Download (falls nötig)
    if cache.exists():
        print(f"\n✅ Cache gefunden: {cache} ({cache.stat().st_size / (1024*1024):.1f} MB)")
    else:
        download_with_progress(CONFIG["url"], str(cache))
    
    # 2. Deutsche Vektoren laden
    all_vectors = load_german_vectors(str(cache))
    
    if len(all_vectors) == 0:
        print("\n❌ Keine deutschen Wörter gefunden. Prüfe die Datei.")
        sys.exit(1)
    
    # 3. Top-Wörter auswählen
    selected = select_top_words(all_vectors, CONFIG["max_words"])
    
    # 4. Dimensionsreduktion
    reduced, explained, pca = reduce_dimensions(selected, CONFIG["target_dims"])
    
    # 5. Qualitätstest
    test_embeddings(reduced)
    
    # 6. JSON exportieren
    meta = export_json(reduced, explained, CONFIG["output_json"], CONFIG["output_meta"])
    
    # 7. Zusammenfassung
    print("═" * 65)
    print("📊 ZUSAMMENFASSUNG")
    print("═" * 65)
    print(f"   Quelle:        ConceptNet Numberbatch 19.08")
    print(f"   Sprache:       Deutsch")
    print(f"   Wörter:        {meta['num_words']:,}")
    print(f"   Dimensionen:   {meta['original_dimensions']} → {meta['dimensions']} (PCA)")
    print(f"   Varianz:       {meta['pca_explained_variance_pct']:.1f}% erhalten")
    print(f"   Dateigröße:    {meta['file_size_bytes'] / (1024*1024):.2f} MB")
    print(f"   Ausgabe:       {CONFIG['output_json']}")
    print(f"   Metadaten:     {CONFIG['output_meta']}")
    print("═" * 65)
    print("\n🎮 Die Datei kann jetzt im Semantris-Spiel verwendet werden!")
    print("   Einbinden per: <script src='german_embeddings.json'></script>")
    print("   Oder per fetch() laden.\n")


if __name__ == "__main__":
    main()


In [ ]:
pip install wordfreq

In [ ]:

"""
======================================================================
 KI-Labor: Deutsche Word-Embeddings aus ConceptNet Numberbatch (v3)
======================================================================

VORAUSSETZUNGEN:
    pip install numpy scikit-learn wordfreq

AUSFUEHRUNG:
    python extract_german_embeddings.py

AENDERUNGEN v3:
    - MUST_INCLUDE mit echten Umlauten (ConceptNet nutzt Unicode)
      aber ss statt sz (ConceptNet normalisiert sz -> ss)
    - Flexionsformen-Filter: entfernt Varianten wenn Grundform
      vorhanden (z.B. "hunde" raus wenn "hund" drin)
    - Bessere Analogie-Ergebnisse durch saubereres Vokabular

LIZENZ: CC-BY-SA 4.0 (Numberbatch), MIT (dieses Skript)
======================================================================
"""

import json
import gzip
import os
import sys
import re
import urllib.request
from pathlib import Path

import numpy as np
from sklearn.decomposition import PCA

try:
    from wordfreq import word_frequency
    HAS_WORDFREQ = True
except ImportError:
    HAS_WORDFREQ = False
    print("  HINWEIS: 'wordfreq' nicht installiert.")
    print("           pip install wordfreq")
    print("           Ohne wordfreq: Fallback auf Datei-Reihenfolge.\n")

# ==================================================================
# KONFIGURATION
# ==================================================================

CONFIG = {
    "url": "https://conceptnet.s3.amazonaws.com/downloads/2019/"
           "numberbatch/numberbatch-19.08.txt.gz",
    "cache_file": "numberbatch-19.08.txt.gz",
    "language": "de",
    "max_words": 5000,
    "target_dims": 100,
    "output_json": "german_embeddings.json",
    "output_meta": "german_embeddings_meta.json",
    "clean_words_only": True,
    "min_word_length": 2,
    "decimal_places": 4,
    # Flexionsformen filtern? (empfohlen fuer Spiel-Einsatz)
    "filter_inflections": True,
}

# ==================================================================
# PFLICHTWOERTER
# ==================================================================
# ConceptNet: echte Umlaute (ä ö ü), aber ss statt ß.
# Also: "könig" (nicht "koenig"), "gross" (nicht "groß")

MUST_INCLUDE = {
    # Tiere
    "hund", "katze", "maus", "pferd", "vogel", "fisch",
    "löwe", "tiger", "elefant", "affe", "schlange", "adler",
    "bär", "wolf", "kuh", "schwein", "huhn", "schaf",
    # Familie
    "mann", "frau", "kind", "junge", "mädchen", "mutter", "vater",
    "bruder", "schwester", "onkel", "tante", "oma", "opa",
    "könig", "königin", "prinz", "prinzessin",
    # Essen
    "brot", "wasser", "milch", "apfel", "kuchen", "eis",
    "fleisch", "obst", "kartoffel", "tomate", "zucker", "käse",
    "reis", "nudel", "salat", "suppe",
    # Natur
    "sonne", "mond", "stern", "regen", "schnee", "wind", "wolke",
    "baum", "blume", "wald", "berg", "meer", "fluss", "see",
    "feuer", "erde", "luft", "himmel",
    # Orte
    "haus", "schule", "kirche", "stadt", "dorf",
    "strasse",  # ConceptNet: ss statt ß
    "krankenhaus", "bibliothek", "kino", "rathaus", "markt",
    # Laender & Staedte
    "deutschland", "frankreich", "italien", "spanien", "england",
    "europa", "amerika", "china", "japan", "russland",
    "berlin", "paris", "london", "rom", "wien", "münchen",
    # Berufe
    "arzt", "lehrer", "polizist", "bauer", "koch", "pilot",
    "richter", "künstler", "soldat", "wissenschaftler",
    # Farben
    "rot", "blau", "grün", "gelb", "schwarz",
    "weiss",  # ConceptNet: ss statt ß
    # Gefuehle
    "angst", "liebe", "freude", "wut", "trauer", "hoffnung",
    "glück",
    # Adjektive
    "gross",  # ConceptNet: ss statt ß
    "klein", "schnell", "langsam", "warm", "kalt",
    "gut", "schlecht", "alt", "neu", "jung", "schön",
    "stark", "schwach", "hell", "dunkel", "laut", "leise",
    "reich", "arm", "klug", "dumm",
    # Verben
    "laufen", "essen", "trinken", "schlafen", "spielen",
    "lernen", "lesen", "schreiben", "denken", "sprechen",
    "arbeiten", "singen", "fliegen", "schwimmen", "kochen",
    # Transport
    "auto", "fahrrad", "zug", "flugzeug", "schiff", "bus",
    # Musik & Kunst
    "musik", "gitarre", "klavier", "lied", "tanz", "bild",
    # Sport
    "fussball",  # ConceptNet: ss statt ß
    "schwimmen", "tennis",
    # Technik
    "computer", "telefon", "internet", "roboter", "maschine",
    "mathematik", "physik", "chemie", "biologie",
    # Szenarien (Eisverkäufer, Sonnenblume)
    "temperatur", "verkauf", "sommer", "winter",
    "sonnenblume", "wachstum", "pflanze", "garten",
    # Schule
    "buch", "stift", "tafel", "klasse", "prüfung", "note",
    # Koerper
    "kopf", "hand", "auge", "herz", "fuss",
    # Zeit
    "tag", "nacht", "morgen", "abend", "stunde", "jahr",
    # Abstraktes
    "zeit", "geld", "frieden", "krieg", "freiheit", "wahrheit",
}


# ==================================================================
# HILFSFUNKTIONEN
# ==================================================================

def download_with_progress(url, filepath):
    """Datei herunterladen mit Fortschrittsanzeige."""
    print(f"\n  Lade herunter:\n   {url}")
    req = urllib.request.Request(url)
    req.add_header("User-Agent", "KI-Labor/3.0")
    response = urllib.request.urlopen(req)
    total = int(response.headers.get("Content-Length", 0))
    downloaded = 0
    with open(filepath, "wb") as f:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                pct = downloaded / total * 100
                print(f"\r   {downloaded/(1024*1024):.1f} / "
                      f"{total/(1024*1024):.1f} MB ({pct:.1f}%)",
                      end="", flush=True)
    print(f"\n   Download abgeschlossen.")


def is_clean_word(word):
    """Nur Buchstaben inkl. echte Umlaute. Kein ß (ConceptNet: ss)."""
    if len(word) < CONFIG["min_word_length"]:
        return False
    # a-z plus ä ö ü (kein ß, da ConceptNet ss nutzt)
    return bool(re.match(r'^[a-zäöü]+$', word))


def get_word_freq(word):
    """Worthaeufigkeit via wordfreq. Probiert auch ß-Variante."""
    if not HAS_WORDFREQ:
        return 0.0
    freq = word_frequency(word, 'de')
    # ConceptNet: "gross" -> wordfreq kennt "groß"
    if freq == 0 and 'ss' in word:
        freq = word_frequency(word.replace('ss', 'ß'), 'de')
    return freq


def is_likely_inflection(word, vocabulary_set):
    """Heuristik: Ist dieses Wort eine Flexionsform eines anderen?

    Prueft ob eine kuerzere Variante (= mutmassliche Grundform)
    bereits im Vokabular ist. Einfache Suffixe werden entfernt.

    Beispiele:
        "hunde" -> "hund" vorhanden? Ja -> Flexion
        "grössere" -> "gross" vorhanden? Nein (anderer Stamm)
        "katzen" -> "katze" vorhanden? Ja -> Flexion
    """
    if len(word) < 4:
        return False

    # Typische deutsche Flexionssuffixe (haeufigste zuerst)
    suffixes = [
        "ungen", "tion", "isch", "lich", "heit", "keit",
        "ern", "ens", "est", "ste",
        "en", "er", "es", "em", "st", "te", "ts",
        "e", "n", "s", "t",
    ]

    for suffix in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            stem = word[:-len(suffix)]
            if stem in vocabulary_set:
                return True

    return False


def filter_inflections(word_freq_list, max_words):
    """Flexionsformen entfernen, Grundformen bevorzugen.

    Strategie: Zuerst die haeufigsten Woerter aufnehmen.
    Wenn ein Wort wie eine Flexionsform eines bereits
    aufgenommenen Worts aussieht, ueberspringen.
    """
    print("   Filtere Flexionsformen...")
    selected = []
    selected_set = set()
    skipped = 0

    for word, freq, vec in word_freq_list:
        if len(selected) >= max_words:
            break

        if is_likely_inflection(word, selected_set):
            skipped += 1
            continue

        selected.append((word, freq, vec))
        selected_set.add(word)

    print(f"   {skipped:,} Flexionsformen uebersprungen")
    return selected


def load_german_vectors(filepath):
    """Deutsche Vektoren aus Numberbatch laden."""
    print(f"\n  Lese Vektoren aus {filepath}...")
    prefix = f"/c/{CONFIG['language']}/"
    vectors = {}
    total_lines = 0
    opener = gzip.open if filepath.endswith(".gz") else open

    with opener(filepath, "rt", encoding="utf-8") as f:
        header = f.readline().strip().split()
        if len(header) == 2:
            print(f"   Modell: {int(header[0]):,} Woerter, "
                  f"{header[1]} Dimensionen")

        for line in f:
            total_lines += 1
            if total_lines % 200000 == 0:
                print(f"\r   {total_lines:,} Zeilen, "
                      f"{len(vectors):,} dt. Woerter...",
                      end="", flush=True)

            if not line.startswith(prefix):
                continue

            parts = line.strip().split(" ")
            if len(parts) < 10:
                continue

            word = parts[0][len(prefix):]
            word = word.replace("_", " ")
            if " " in word:
                continue
            if CONFIG["clean_words_only"] and not is_clean_word(word):
                continue

            try:
                vec = np.array([float(x) for x in parts[1:]],
                               dtype=np.float32)
                vectors[word] = vec
            except ValueError:
                continue

    print(f"\n   {len(vectors):,} deutsche Woerter geladen")
    return vectors


def select_top_words(vectors, max_words):
    """Top-Woerter per Frequenz + Flexions-Filter auswaehlen."""
    print(f"\n  Waehle die {max_words} wichtigsten Woerter...")

    if HAS_WORDFREQ:
        print("   Methode: wordfreq Haeufigkeitsranking")
    else:
        print("   Methode: Datei-Reihenfolge (Fallback)")

    # --- Pflichtwoerter sichern ---
    must_have = {}
    for target in MUST_INCLUDE:
        if target in vectors:
            must_have[target] = vectors[target]

    found_must = len(must_have)
    missing_must = MUST_INCLUDE - set(must_have.keys())
    print(f"   Pflicht-Woerter: {found_must}/{len(MUST_INCLUDE)}")
    if missing_must:
        print(f"   Nicht im Modell: {', '.join(sorted(missing_must))}")

    # --- Restliche Woerter nach Haeufigkeit sortieren ---
    remaining = {w: v for w, v in vectors.items() if w not in must_have}

    if HAS_WORDFREQ:
        print(f"   Berechne Haeufigkeiten fuer "
              f"{len(remaining):,} Woerter...")
        scored = []
        for i, (word, vec) in enumerate(remaining.items()):
            if i % 100000 == 0 and i > 0:
                print(f"\r   {i:,} bewertet...", end="", flush=True)
            scored.append((word, get_word_freq(word), vec))

        scored.sort(key=lambda x: x[1], reverse=True)
        print(f"\r   {len(scored):,} Woerter bewertet.              ")

        has_freq = sum(1 for _, f, _ in scored if f > 0)
        print(f"   Davon mit bekannter Haeufigkeit: {has_freq:,}")

        print("\n   Top-15 haeufigste Woerter:")
        for w, f, _ in scored[:15]:
            print(f"     {w:20s}  freq={f:.7f}")
    else:
        scored = [(w, 0.0, v) for w, v in remaining.items()]

    # --- Flexionsformen filtern ---
    remaining_slots = max_words - len(must_have)

    if CONFIG["filter_inflections"] and HAS_WORDFREQ:
        filtered = filter_inflections(scored, remaining_slots + 1000)
        # +1000 Puffer, da manche Grundformen auch rausfliegen
    else:
        filtered = scored

    # --- Auffuellen ---
    selected = dict(must_have)
    count = 0
    for word, freq, vec in filtered:
        if count >= remaining_slots:
            break
        selected[word] = vec
        count += 1

    if HAS_WORDFREQ:
        # Grenzbereich anzeigen
        boundary = filtered[max(0, count-5):count]
        print(f"\n   Grenzbereich (letzte aufgenommene):")
        for w, f, _ in boundary:
            print(f"     {w:20s}  freq={f:.7f}")

    print(f"\n   {len(selected):,} Woerter ausgewaehlt")
    return selected


def reduce_dimensions(vectors, target_dims):
    """PCA-Dimensionsreduktion."""
    dim_orig = len(list(vectors.values())[0])
    print(f"\n  PCA: {dim_orig} -> {target_dims} Dimensionen...")

    words = list(vectors.keys())
    matrix = np.array([vectors[w] for w in words])
    mean = matrix.mean(axis=0)
    matrix_centered = matrix - mean

    pca = PCA(n_components=target_dims)
    matrix_reduced = pca.fit_transform(matrix_centered)

    explained = float(sum(pca.explained_variance_ratio_) * 100)
    print(f"   Erklaerte Varianz: {explained:.1f}%")

    norms = np.linalg.norm(matrix_reduced, axis=1, keepdims=True)
    norms[norms == 0] = 1
    matrix_normalized = matrix_reduced / norms

    result = {w: matrix_normalized[i] for i, w in enumerate(words)}
    print(f"   Reduktion abgeschlossen")
    return result, explained, pca


def export_json(vectors, explained_variance, output_path, meta_path):
    """Als JSON exportieren."""
    print(f"\n  Exportiere nach {output_path}...")
    dp = CONFIG["decimal_places"]

    export = {}
    for word, vec in sorted(vectors.items()):
        export[word] = [round(float(v), dp) for v in vec]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(export, f, ensure_ascii=False, separators=(",", ":"))

    file_size = os.path.getsize(output_path)
    print(f"   Dateigroesse: {file_size / (1024*1024):.2f} MB")
    print(f"   Woerter: {len(export):,}")
    print(f"   Dimensionen: {len(list(export.values())[0])}")

    meta = {
        "source": "ConceptNet Numberbatch 19.08",
        "license": "CC-BY-SA 4.0",
        "language": CONFIG["language"],
        "num_words": len(export),
        "dimensions": CONFIG["target_dims"],
        "original_dimensions": 300,
        "pca_explained_variance_pct": round(float(explained_variance), 2),
        "decimal_places": dp,
        "file_size_bytes": file_size,
        "word_selection": "wordfreq" if HAS_WORDFREQ else "file_order",
        "inflection_filter": CONFIG["filter_inflections"],
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"   Export abgeschlossen")
    return meta


def test_embeddings(vectors):
    """Embedding-Qualitaet testen."""
    print("\n  Qualitaetstest: Aehnlichste Woerter\n")

    words = list(vectors.keys())
    matrix = np.array([vectors[w] for w in words])

    def most_similar(word, n=8):
        if word not in vectors:
            return f"   '{word}' nicht im Vokabular"
        query = vectors[word]
        sims = matrix @ query
        indices = np.argsort(sims)[::-1]
        results = []
        for idx in indices:
            w = words[idx]
            if w != word:
                results.append((w, float(sims[idx])))
            if len(results) >= n:
                break
        return results

    def analogy(a, b, c):
        if a not in vectors or b not in vectors or c not in vectors:
            missing = [w for w in [a, b, c] if w not in vectors]
            return f"   Nicht gefunden: {', '.join(missing)}"
        result_vec = vectors[b] - vectors[a] + vectors[c]
        norm = np.linalg.norm(result_vec)
        if norm > 0:
            result_vec = result_vec / norm
        sims = matrix @ result_vec
        indices = np.argsort(sims)[::-1]
        exclude = {a, b, c}
        results = []
        for idx in indices:
            w = words[idx]
            if w not in exclude:
                results.append((w, float(sims[idx])))
            if len(results) >= 5:
                break
        return results

    test_words = ["hund", "sonne", "berlin", "arzt", "musik", "auto"]
    for tw in test_words:
        result = most_similar(tw)
        if isinstance(result, str):
            print(result)
        else:
            top5 = ", ".join(f"{w}({s:.3f})" for w, s in result[:5])
            print(f"   {tw:15s} -> {top5}")

    print("\n  Analogie-Test: A ist zu B wie C ist zu ???\n")
    analogies = [
        ("mann", "könig", "frau", "-> königin?"),
        ("deutschland", "berlin", "frankreich", "-> paris?"),
        ("hund", "katze", "gross", "-> klein?"),
        ("sommer", "warm", "winter", "-> kalt?"),
        ("könig", "königin", "prinz", "-> prinzessin?"),
    ]
    for a, b, c, label in analogies:
        result = analogy(a, b, c)
        if isinstance(result, str):
            print(f"   {a} -> {b}, {c} -> ? {result}")
        else:
            top = ", ".join(f"{w}({s:.3f})" for w, s in result[:3])
            print(f"   {a} -> {b}, {c} -> ? {label}")
            print(f"   {'':15s}   Ergebnis: {top}")
    print()


# ==================================================================
# HAUPTPROGRAMM
# ==================================================================

def main():
    print("=" * 65)
    print("  KI-Labor: Deutsche Word-Embeddings Extraktor v3")
    print("  ConceptNet Numberbatch -> Kompaktes JSON")
    print("=" * 65)

    cache = Path(CONFIG["cache_file"])
    if cache.exists():
        mb = cache.stat().st_size / (1024 * 1024)
        print(f"\n  Cache gefunden: {cache} ({mb:.1f} MB)")
    else:
        download_with_progress(CONFIG["url"], str(cache))

    all_vectors = load_german_vectors(str(cache))
    if not all_vectors:
        print("\n  Keine deutschen Woerter gefunden!")
        sys.exit(1)

    selected = select_top_words(all_vectors, CONFIG["max_words"])
    reduced, explained, pca = reduce_dimensions(
        selected, CONFIG["target_dims"])
    test_embeddings(reduced)
    meta = export_json(reduced, explained,
                       CONFIG["output_json"], CONFIG["output_meta"])

    print("=" * 65)
    print("  ZUSAMMENFASSUNG")
    print("=" * 65)
    print(f"   Quelle:       ConceptNet Numberbatch 19.08")
    print(f"   Woerter:      {meta['num_words']:,}")
    print(f"   Dimensionen:  300 -> {meta['dimensions']} (PCA)")
    print(f"   Varianz:      {meta['pca_explained_variance_pct']:.1f}%")
    print(f"   Dateigroesse: {meta['file_size_bytes']/(1024*1024):.2f} MB")
    print(f"   Auswahl:      {meta['word_selection']}")
    print(f"   Flexions-Filter: {meta['inflection_filter']}")
    print(f"   Ausgabe:      {CONFIG['output_json']}")
    print("=" * 65)
    print(f"\n  Fertig! JSON kann per fetch() in die App geladen werden.\n")


if __name__ == "__main__":
    main()